# Notebook to test helper functions independently

## Testing Product list generator helper functions below

In [1]:
from PIL import Image
from image_utils import encode_image_to_base64

# valid case
img = Image.new("RGB", (10, 10), color="red")
encoded = encode_image_to_base64(img)
print("✓ valid image:", encoded[:30], "...")

# invalid case
try:
    encode_image_to_base64(None)
except ValueError as e:
    print("✓ correctly caught:\n", e)

✓ valid image: /9j/4AAQSkZJRgABAQAAAQABAAD/2w ...
✓ correctly caught:
 [encode_image_to_base64] ValueError: received None instead of an image.
Context: check the 'image' column of the product row.
Suggestion: verify the dataset loaded correctly before encoding.


In [2]:
from prompt_builder import create_product_listing_prompt

# valid case
fake_product = {"productDisplayName": "Test Shirt", "masterCategory": "Apparel"}
prompt = create_product_listing_prompt(fake_product)
print("✓ valid product:", prompt[:80], "...")

# invalid cases
try:
    create_product_listing_prompt(None)
except ValueError as e:
    print("✓ correctly caught None:\n", e)

try:
    create_product_listing_prompt(["not", "a", "dict"])
except ValueError as e:
    print("✓ correctly caught wrong type:\n", e)

✓ valid product: 
    Role: You are an expert e-commerce copywriter. 
    Task: Analyze the produ ...
✓ correctly caught None:
 [create_product_listing_prompt] ValueError: product is None.
Context: expected a dict-like row with product fields.
Suggestion: check that the dataframe row was passed correctly.
✓ correctly caught wrong type:
 [create_product_listing_prompt] ValueError: product of type list has no .get() method.
Context: expected a dict or pandas Series.
Suggestion: pass a single row (e.g. df.iloc[i]), not a DataFrame or list.


In [3]:
from data_loader import load_product_dataset

# valid case
df = load_product_dataset()
print(f"✓ loaded {len(df)} rows, columns: {df.columns.to_list()}")

# invalid case — bad dataset name
try:
    load_product_dataset(dataset_name="not/a-real-dataset")
except RuntimeError as e:
    print("✓ correctly caught:\n", e)

/opt/anaconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ loaded 50 rows, columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']
✓ correctly caught:
 [load_product_dataset] DatasetNotFoundError: Dataset 'not/a-real-dataset' doesn't exist on the Hub or cannot be accessed.
Context: failed to load 'not/a-real-dataset' (split='train[:50]') from HuggingFace.
Suggestion: check internet connection, HF Hub status, or that the 'datasets' package is installed and up to date.


In [4]:
import os
from config import validate_config

# valid case (assuming .env is set up correctly)
try:
    validate_config()
    print("✓ config valid")
except RuntimeError as e:
    print("unexpected failure:\n", e)

# invalid case — simulate missing key
os.environ.pop("OPENAI_API_KEY", None)
import importlib
import config
importlib.reload(config)  # re-run module-level code with key removed
try:
    config.validate_config()
except RuntimeError as e:
    print("✓ correctly caught:\n", e)

✓ config valid


## Testing meal planner helper functions

In [1]:
from json_utils import strip_markdown_json_fences, parse_json_response
import json

# valid — plain JSON
print("✓ plain JSON:", parse_json_response('{"score": 1, "reason": "ok"}'))

# valid — fenced JSON
fenced = '```json\n{"score": 1, "reason": "ok"}\n```'
print("✓ fenced JSON:", parse_json_response(fenced))

# invalid — malformed JSON
try:
    parse_json_response('{"score": 1, "reason": }')
except json.JSONDecodeError as e:
    print("✓ correctly caught:\n", e)

# invalid — None input
try:
    strip_markdown_json_fences(None)
except ValueError as e:
    print("✓ correctly caught:\n", e)

✓ plain JSON: {'score': 1, 'reason': 'ok'}
✓ fenced JSON: {'score': 1, 'reason': 'ok'}
✓ correctly caught:
 [parse_json_response] JSONDecodeError: Expecting value
Context: failed at line 1, column 24 of cleaned LLM response.
Raw content (first 200 chars): {"score": 1, "reason": }
Suggestion: check the LLM's response_format/prompt, or inspect raw output.: line 1 column 24 (char 23)
✓ correctly caught:
 [strip_markdown_json_fences] ValueError: content is None.
Context: expected a string LLM response.
Suggestion: check that the LLM call actually returned a response before passing it here.


In [2]:
import os, importlib
import config

# valid case (assumes .env has OPENAI_API_KEY set)
try:
    config.validate_config()
    print("✓ config valid")
except RuntimeError as e:
    print("unexpected failure:\n", e)

# invalid case — simulate missing key
saved = os.environ.pop("OPENAI_API_KEY", None)
importlib.reload(config)
try:
    config.validate_config()
except RuntimeError as e:
    print("✓ correctly caught:\n", e)
if saved:
    os.environ["OPENAI_API_KEY"] = saved
    importlib.reload(config)

✓ config valid


In [3]:
from pydantic import ValidationError
from models import Ingredient, Recipe

# valid case
ing = Ingredient(name="chicken breast", quantity="500g")
print("✓ valid ingredient:", ing)

# invalid case — wrong ingredient count (needs 5, only 1 given)
try:
    Recipe(day="Monday", theme="Italian", dish_name="Pasta",
           ingredients=[ing], instructions="Cook it")
except ValidationError as e:
    print("✓ correctly caught:\n", e)

✓ valid ingredient: name='chicken breast' quantity='500g'
✓ correctly caught:
 1 validation error for Recipe
ingredients
  Value error, Recipe must have exactly 5 ingredients, got 1 [type=value_error, input_value=[Ingredient(name='chicken...east', quantity='500g')], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [4]:
from unittest.mock import MagicMock
from meal_plan_generator import MealPlanGenerator

gen = MealPlanGenerator()

# valid case — mock a well-formed response
fake_response = MagicMock()
fake_response.content = '{"dietary_preference": "omnivore", "recipes": [...], "shopping_list": [...]}'
# (use a real valid payload in practice; abbreviated here for illustration)

# invalid case — malformed JSON
try:
    gen._parse_and_validate('{"dietary_preference": "omnivore", }')
except Exception as e:
    print("✓ correctly caught malformed JSON:\n", e)

# invalid case — valid JSON, wrong schema (missing recipes)
try:
    gen._parse_and_validate('{"dietary_preference": "omnivore", "recipes": [], "shopping_list": []}')
except Exception as e:
    print("✓ correctly caught schema violation:\n", e)

✓ correctly caught malformed JSON:
 [parse_json_response] JSONDecodeError: Expecting property name enclosed in double quotes
Context: failed at line 1, column 36 of cleaned LLM response.
Raw content (first 200 chars): {"dietary_preference": "omnivore", }
Suggestion: check the LLM's response_format/prompt, or inspect raw output.: line 1 column 36 (char 35)
✓ correctly caught schema violation:
 1 validation error for MealPlan
recipes
  Value error, Meal plan must have exactly 7 recipes, got 0 [type=value_error, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [5]:
from email_utils import send_meal_plan_email

fake_plan = {"dietary_preference": "omnivore", "shopping_list": [], "recipes": []}

# invalid case — bad credentials (expect SMTPAuthenticationError)
result = send_meal_plan_email("test@example.com", fake_plan)
print(result["success"], "\n", result["message"])

False 
 [send_meal_plan_email] SMTPResponseException: (334, b'UGFzc3dvcmQ6')
Suggestion: check recipient address 'test@example.com' and SMTP server status.
